In [2]:
from typing import Callable

import jax.experimental
import jax.experimental.sparse
import soldis as sd
import jax
import jax.numpy as jnp
from jax import Array

jax.config.update("jax_enable_x64", True)


def f(x: Array, _: None) -> Array:
    return x * x - 2.0


def afunc(x: Array, solver: sd.newton.NewtonSolver) -> Array:
    return solver.root(x, None).value


def jvp(location: Array, _: None) -> Callable[[Array], Array]:
    def wrapper(x: Array) -> Array:
        return jax.jvp(lambda y: f(y, None), (location,), (x,))[1]

    return wrapper


solver = sd.newton.NewtonSolver(f, lin_solver=sd.linear.DirectLinearSolver())
jax.jit(afunc)(jnp.asarray([1.0]), solver)

solver_cg = sd.newton.NewtonSolver(f, lin_solver=sd.linear.CG())
solver_cg.root(jnp.asarray([1.0]), None)

SolverState(value=Array([1.41421356], dtype=float64), args=None, residual=Array([4.51083505e-12], dtype=float64), iteration=Array(4, dtype=int64, weak_type=True), converged=Array(True, dtype=bool))